In [5]:
"""
Hand Generation Dataset Sampler
================================
Selects best 2000 images from FreiHAND + 1000 from 11K Hands
Scoring based on: sharpness + pose diversity (via clustering)

Requirements:
    pip install opencv-python numpy scikit-learn tqdm Pillow
"""

import os
import json
import cv2
import numpy as np
from tqdm import tqdm
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
import shutil
from collections import defaultdict

# ─────────────────────────────────────────
#  PATHS  (Kaggle)
# ─────────────────────────────────────────
FREIHAND_RGB_DIR = "/kaggle/input/datasets/danieldelro/freihand/evaluation/rgb"
FREIHAND_XYZ     = "/kaggle/input/datasets/danieldelro/freihand/evaluation_xyz.json"
HANDK_DIR        = "/kaggle/input/datasets/shyambhu/hands-and-palm-images-dataset/Hands/Hands"
OUTPUT_DIR       = "/kaggle/working/hand_gen_dataset"

N_FREI      = 0
N_11K       = 3000
N_CLUSTERS  = 50   # pose diversity buckets


# ─────────────────────────────────────────
#  SHARPNESS: Laplacian Variance
# ─────────────────────────────────────────
def sharpness_score(img_path: str) -> float:
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return 0.0
    return float(cv2.Laplacian(img, cv2.CV_64F).var())


# ─────────────────────────────────────────
#  POSE FEATURE: FreiHAND 3D keypoints
# ─────────────────────────────────────────
def pose_feature_freihand(keypoints_3d: list) -> np.ndarray:
    kp = np.array(keypoints_3d)   # (21, 3)
    kp = kp - kp[0]               # root-relative
    return kp.flatten()           # (63,)


# ─────────────────────────────────────────
#  POSE FEATURE: HOG proxy (11K / fallback)
# ─────────────────────────────────────────
def pose_feature_hog(img_path: str) -> np.ndarray:
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return np.zeros(1764)
    img = cv2.resize(img, (64, 64))
    hog = cv2.HOGDescriptor(
        _winSize=(64, 64), _blockSize=(16, 16),
        _blockStride=(8, 8), _cellSize=(8, 8), _nbins=9
    )
    return hog.compute(img).flatten()


# ─────────────────────────────────────────
#  DIVERSITY SAMPLING
#  Clusters images by pose, then round-robins
#  picking the sharpest from each cluster
# ─────────────────────────────────────────
def diversity_sample(entries: list, n: int, n_clusters: int) -> list:
    features = np.stack([e["pose_feat"] for e in entries])
    features = normalize(features)

    k = min(n_clusters, len(entries))
    print(f"  Clustering into {k} pose buckets...")
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(features)

    clusters = defaultdict(list)
    for e, lbl in zip(entries, labels):
        e["cluster"] = int(lbl)
        clusters[int(lbl)].append(e)

    for c in clusters:
        clusters[c].sort(key=lambda x: x["sharpness"], reverse=True)

    selected, iters = [], {c: iter(v) for c, v in clusters.items()}
    active = list(iters.keys())
    while len(selected) < n and active:
        for c in list(active):
            try:
                selected.append(next(iters[c]))
                if len(selected) == n:
                    break
            except StopIteration:
                active.remove(c)

    return selected[:n]


# ─────────────────────────────────────────
#  LOAD & SCORE: FreiHAND
# ─────────────────────────────────────────
def load_freihand() -> list:
    all_xyz = []
    if os.path.exists(FREIHAND_XYZ):
        print("  Loading keypoints from evaluation_xyz.json...")
        with open(FREIHAND_XYZ) as f:
            all_xyz = json.load(f)
    else:
        print(f"  WARNING: {FREIHAND_XYZ} not found — using HOG fallback for pose")

    img_files = sorted([
        f for f in os.listdir(FREIHAND_RGB_DIR)
        if f.lower().endswith((".jpg", ".png"))
    ])

    entries = []
    print(f"  Scoring {len(img_files)} FreiHAND images...")
    for i, fname in enumerate(tqdm(img_files)):
        path  = os.path.join(FREIHAND_RGB_DIR, fname)
        sharp = sharpness_score(path)
        pose  = pose_feature_freihand(all_xyz[i]) if (all_xyz and i < len(all_xyz)) \
                else pose_feature_hog(path)
        entries.append({"path": path, "sharpness": sharp, "pose_feat": pose, "source": "freihand"})
    return entries


# ─────────────────────────────────────────
#  LOAD & SCORE: 11K Hands
# ─────────────────────────────────────────
def load_11k() -> list:
    img_files = sorted([
        os.path.join(HANDK_DIR, f)
        for f in os.listdir(HANDK_DIR)
        if f.lower().endswith((".jpg", ".png"))
    ])

    entries = []
    print(f"  Scoring {len(img_files)} 11K Hands images...")
    for path in tqdm(img_files):
        sharp = sharpness_score(path)
        pose  = pose_feature_hog(path)
        entries.append({"path": path, "sharpness": sharp, "pose_feat": pose, "source": "11k"})
    return entries


# ─────────────────────────────────────────
#  SAVE: output as 00000.jpg, 00001.jpg ...
#  FreiHAND first (00000–01999)
#  11K Hands next  (02000–02999)
# ─────────────────────────────────────────
def save_all(frei_selected: list, handk_selected: list) -> list:
    img_out = os.path.join(OUTPUT_DIR, "images")
    os.makedirs(img_out, exist_ok=True)

    manifest = []
    counter  = 0

    for e in tqdm(frei_selected + handk_selected, desc="Saving"):
        ext  = os.path.splitext(e["path"])[1].lower() or ".jpg"
        dest = os.path.join(img_out, f"{counter:05d}{ext}")
        shutil.copy2(e["path"], dest)
        manifest.append({
            "filename"  : f"{counter:05d}{ext}",
            "source"    : e["source"],
            "original"  : e["path"],
            "sharpness" : round(e["sharpness"], 4),
            "cluster"   : e.get("cluster", -1)
        })
        counter += 1

    return manifest


# ─────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("\n=== FreiHAND ===")
    frei_entries  = load_freihand()
    frei_selected = diversity_sample(frei_entries, N_FREI, N_CLUSTERS)

    print("\n=== 11K Hands ===")
    handk_entries  = load_11k()
    handk_selected = diversity_sample(handk_entries, N_11K, N_CLUSTERS)

    print("\n=== Saving ===")
    manifest = save_all(frei_selected, handk_selected)

    manifest_path = os.path.join(OUTPUT_DIR, "dataset_manifest.json")
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)

    frei_sharp  = np.mean([e["sharpness"] for e in frei_selected])
    handk_sharp = np.mean([e["sharpness"] for e in handk_selected])

    print("\n" + "="*50)
    print("  DATASET SUMMARY")
    print("="*50)
    print(f"  FreiHAND  : {len(frei_selected):,} images  (avg sharpness: {frei_sharp:.1f})")
    print(f"  11K Hands : {len(handk_selected):,} images  (avg sharpness: {handk_sharp:.1f})")
    print(f"  Total     : {len(manifest):,} images")
    print(f"  Naming    : 00000.jpg → {len(manifest)-1:05d}.jpg")
    print(f"  Saved to  : {OUTPUT_DIR}/images/")
    print(f"  Manifest  : {manifest_path}")
    print("="*50)


if __name__ == "__main__":
    main()


=== FreiHAND ===
  Scoring 3960 FreiHAND images...


100%|██████████| 3960/3960 [00:13<00:00, 291.33it/s]


  Clustering into 50 pose buckets...

=== 11K Hands ===
  Scoring 11076 11K Hands images...


100%|██████████| 11076/11076 [03:39<00:00, 50.40it/s]


  Clustering into 50 pose buckets...

=== Saving ===


Saving: 100%|██████████| 3000/3000 [00:07<00:00, 408.45it/s]


  DATASET SUMMARY
  FreiHAND  : 500 images  (avg sharpness: 1481.6)
  11K Hands : 2,500 images  (avg sharpness: 90.7)
  Total     : 3,000 images
  Naming    : 00000.jpg → 02999.jpg
  Saved to  : /kaggle/working/hand_gen_dataset/images/
  Manifest  : /kaggle/working/hand_gen_dataset/dataset_manifest.json


In [6]:
import shutil
shutil.make_archive("/kaggle/working/hand_easy", "zip", "/kaggle/working/hand_gen_dataset")
print("Done! Download hand_dataset.zip from the Output tab")

Done! Download hand_dataset.zip from the Output tab
